In [5]:
from qiskit import QuantumCircuit, transpile
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import Sampler
from qiskit.result import counts
from qiskit_aer import Aer
import numpy as np
import math
from qiskit.circuit.library import QFT
from qiskit.circuit.library.standard_gates import U1Gate
from qiskit import ClassicalRegister
import matplotlib.pyplot as plt
from math import gcd
from fractions import Fraction
import time

service = QiskitRuntimeService(channel="ibm_quantum", token="43ed869847a23fa74c73d33739f093a84736361f7f95bccbac355a0449501533339b0c9c51abcc8adfda78f6310ca61522e5e7119969e573479e6056e824269d")

# Define the modular exponentiation gate
def controlled_modular_exponentiation(qc, control, target, a, exponent, modulus):
    qc.append(U1Gate((2 * np.pi * a**exponent) / modulus).control(), [control, target])

# Define the function to create the initial state
def create_initial_state(L):
    # Calculate the value of t
    t = 2 * L + 2
    
    # Create a quantum circuit with t+L qubits and L classical bits for measurements
    qc = QuantumCircuit(t + L, t)
    
    # Apply X gate to flip the second register to |1⟩ state
    qc.x(range(t, t + L))
    
    # Apply Hadamard gate to all qubits in the first register
    qc.h(range(t))
    
    return qc

# Function to swap the first half and the second half of qubits
def swap_first_and_second_half(qc, t):
    for i in range(L//2):
        qc.swap(i, i + L//2)

# Function to reset the first half of qubits to |0⟩ state
def reset_first_half(qc, t):
    for i in range(t//2):
        qc.reset(i)

def apply_controlled_modular_exponentiation_to_computerA(qc, t, a, modulus):
    for qubit in range(t//2):
        controlled_modular_exponentiation(qc, qubit, t, a, qubit, modulus)

def apply_controlled_modular_exponentiation_to_computerB(qc, t, a, modulus):
    for qubit in range(t//2, t):
        controlled_modular_exponentiation(qc, qubit, t, a, qubit, modulus)


def continued_fractions_algorithm(x, epsilon=1e-6):
    fractions = []
    while abs(x) >= epsilon:
        integer_part = math.floor(x)
        fractions.append(integer_part)
        if abs(x - integer_part) < epsilon:
            break  # Break the loop if the fractional part is within epsilon
        x = 1 / (x - integer_part)
    return fractions

def best_r_from_continued_fractions(fractions):
    n0, n1 = fractions[0], fractions[1]
    for i in range(2, len(fractions)):
        n2 = fractions[i] * n1 + n0
        n0, n1 = n1, n2
    return n1

def post_process_factors(r, a, N):

    # Check if r is odd
    if r % 2 != 0:
        return None  # r is odd, cannot continue
    
    # Calculate x
    x = pow(a, r // 2, N)
    
    # Find factors
    p = gcd(x + 1, N)
    q = gcd(x - 1, N)
    
    if p != 1 and p != N:
        return (p, N // p)
    elif q != 1 and q != N:
        return (q, N // q)
    else:
        return None  # Factor not found, try a different 'a'

# Main function to run the factoring algorithm
def run_shor_algorithm(L, a, modulus, max_iterations=1000):
    # Load your IBM Quantum account
    backend = service.least_busy(simulator=False, operational=True)
    factors_found = False
    iterations = 0

    start_time = time.time()
    while not factors_found and iterations < max_iterations:
        iterations += 1
        print("Iteration:", iterations)
        
        # Create initial state circuit
        initial_state_circuit = create_initial_state(L)

        # Calculate the value of t
        t = 2 * L + 2

        # Apply controlled modular exponentiation gate to every qubit in the first half of the register
        apply_controlled_modular_exponentiation_to_computerA(initial_state_circuit, L, a, modulus)

        # Apply Inverse QFT to the first half of the qubits
        qft_dagger = QFT(num_qubits=L//2, inverse=True)
        initial_state_circuit.append(qft_dagger, range(L//2))

        # Measure the first half of the register
        initial_state_circuit.measure(range(L//2), range(L//2))
        
        # Swap the first half and the second half of qubits
        swap_first_and_second_half(initial_state_circuit, L//2)
                
        # Measure the time for the first segment
        mid_time = time.time()
        first_segment_time = mid_time - start_time
        print("Time for first segment:", first_segment_time, "seconds")
        
        # Reset the first half of the qubits to |0⟩ state
        reset_first_half(initial_state_circuit, L//2)

        # Apply controlled modular exponentiation gate to every qubit in the second half of the register
        apply_controlled_modular_exponentiation_to_computerB(initial_state_circuit, L, a, modulus)

        # Apply Inverse QFT to the second half of the qubits
        qft_dagger = QFT(num_qubits=L//2, inverse=True)
        initial_state_circuit.append(qft_dagger, range(L//2, L))

        # Measure the second half of the register
        initial_state_circuit.measure(range(L//2, L), range(L//2, L))

        # Measure the time for the second segment
        end_time = time.time()
        second_segment_time = end_time - mid_time
        print("Time for second segment:", second_segment_time, "seconds")
        
        # Transpile the circuit
        transpiled_circuit = transpile(initial_state_circuit, backend=backend)

        sampler = Sampler(mode=backend)
        job = sampler.run([transpiled_circuit])
        result = job.result()
        counts = result.counts()
        
        # Extract measurement outcome
        measurement_outcomes = list(counts.keys())
        m_decimal = int(measurement_outcomes[0], 2)
        m = m_decimal / (2**(t))

        # Apply continued fractions algorithm to obtain r
        fractions = continued_fractions_algorithm(m)
        r = best_r_from_continued_fractions(fractions)
        print("Fractions:", fractions)
        print("Estimated r:", r)

        # Post-process to find factors
        factors = post_process_factors(r, a, modulus)
        print("Factors:", factors)

        if factors is not None:
            factors_found = True
            print("Factors found:", factors)
        else:
            print("Factors not found, trying again...")

    total_time = end_time - start_time
    print("Total Elapsed Time:", total_time, "seconds")
    
    if not factors_found:
        print("Max iterations reached. Factors not found.")

# Example usage
L = 4
a = 2  # Choose the base 'a' of the modular exponentiation
modulus = 15  # The modulus for the modular exponentiation

run_shor_algorithm(L, a, modulus)

Iteration: 1
Time for first segment: 0.002270936965942383 seconds
Time for second segment: 0.0016939640045166016 seconds


KeyboardInterrupt: 